# **Modeling & Feature Benchmarks**
Compare model architectures and feature sets.
- Establish competitive Naive Baseline.
- Test optimized Linear model (Ridge).
- Evaluate Champion model (XGBoost).
- Validate using strictly isolated TimeSeriesSplit (No Leakage).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
import yaml
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Plotting config
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 6)})

# Create figures dir
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)


# Metric: RMSPE
def rmspe(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    y_pred = np.maximum(y_pred, 1.0)  # Stability
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))


# Load config
CONFIG_PATH = Path("../configs/params.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)


## **1. Load & Standardize Data**
Filter open days with sales > 0. Standardize DayOfWeek to business standard (1-7). Sort by date for chronological validation. Isolate holdout (4 weeks) and simulation (6 weeks) sets before cross-validation to guarantee zero future data leaks during model testing.


In [ ]:
# Load
train_df = pd.read_csv(f"../{config['data']['raw_train']}", low_memory=False)
store_df = pd.read_csv(f"../{config['data']['raw_store']}")
df = pd.merge(train_df, store_df, on="Store", how="left")

# Filter
df = df[(df["Open"] == 1) & (df["Sales"] > 0)].copy()

# Temporal Alignment
df["Date"] = pd.to_datetime(df["Date"])
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1  # 1=Mon, 7=Sun
df.sort_values("Date", inplace=True, ignore_index=True)

# Strict Split: Simulation -> Holdout -> CV
max_date = df["Date"].max()
sim_start = max_date - pd.Timedelta(days=config["data_split"]["simulation_days"])
holdout_start = sim_start - pd.Timedelta(days=config["data_split"]["holdout_days"])

df_sim = df[df["Date"] > sim_start].copy()
df_holdout = df[(df["Date"] > holdout_start) & (df["Date"] <= sim_start)].copy()
df_cv = df[df["Date"] <= holdout_start].copy()  # Used for CV tuning

print(f"Total Dataset: {df['Date'].min().date()} to {max_date.date()}")
print(f"CV Training Pool: {len(df_cv)} rows")
print(f"Holdout Set: {len(df_holdout)} rows")
print(f"Simulation Set: {len(df_sim)} rows (Hidden until drift tests)")


## **2. Naive Baseline: Store Median**
Establish the floor. Predict every store's sales as its historical median. This handles store-specific variance without any ML.

In [ ]:
# Train-only TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=config["train"]["n_splits"])
naive_results = []

for train_idx, test_idx in tscv.split(df_cv):
    fold_train, fold_test = df_cv.iloc[train_idx], df_cv.iloc[test_idx]
    y_true = fold_test["Sales"]

    # Predict as store median
    store_medians = fold_train.groupby("Store")["Sales"].median()
    preds = fold_test["Store"].map(store_medians).fillna(fold_train["Sales"].median())

    naive_results.append(
        {
            "RMSPE": rmspe(y_true, preds),
            "RMSE": np.sqrt(mean_squared_error(y_true, preds)),
            "MAE": mean_absolute_error(y_true, preds),
            "R2": r2_score(y_true, preds),
        }
    )

# Format for summary
naive_avg = pd.DataFrame(naive_results).mean().to_dict()
print(
    f"Naive Baseline - RMSPE: {naive_avg['RMSPE'] * 100:.2f}%, R2: {naive_avg['R2']:.3f}"
)


## **3. Feature Engineering: Target Encoding & Advance Features**
Extract date components. Encode categoricals. Use Out-of-Fold (OOF) Target Encoding for Store ID to prevent data leakage.

In [ ]:
def build_features_v4(train_df, test_df):
    df_full = pd.concat([train_df, test_df], axis=0)

    # 1. Date Components
    df_full["Year"] = df_full["Date"].dt.year
    df_full["Month"] = df_full["Date"].dt.month
    df_full["WeekOfYear"] = df_full["Date"].dt.isocalendar().week.astype(int)

    # 2. Categoricals (OHE)
    df_full["StateHoliday"] = df_full["StateHoliday"].astype(str)
    df_full = pd.get_dummies(
        df_full, columns=["StoreType", "Assortment", "StateHoliday"]
    )

    # 3. Competition Logic - Impute using Train Median ONLY (No Leakage)
    train_comp_median = train_df["CompetitionDistance"].median()
    df_full["LogCompDist"] = np.log1p(
        df_full["CompetitionDistance"].fillna(train_comp_median)
    )

    # Split back
    train_out = df_full.iloc[: len(train_df)].copy()
    test_out = df_full.iloc[len(train_df) :].copy()

    # 4. Sequential Target Encoding (Expanding Mean) - Zero Leakage
    train_out["Store_TargetMean"] = train_out.groupby("Store")["Sales"].transform(
        lambda x: x.shift().expanding().mean()
    )

    # Handle Cold Start
    global_mean = train_out["Sales"].mean()
    train_out["Store_TargetMean"] = train_out["Store_TargetMean"].fillna(global_mean)

    # Test mapping: Uses the final average levels from the training set
    final_store_means = train_out.groupby("Store")["Sales"].mean()
    test_out["Store_TargetMean"] = (
        test_out["Store"].map(final_store_means).fillna(global_mean)
    )

    return train_out, test_out


# Preview
df_train, df_test_mock = build_features_v4(df_cv.iloc[:10000], df_cv.iloc[10000:12000])
print(f"Features engineered. Total columns: {len(df_train.columns)}")


## **4. Fair Linear Comparison: Ridge**
Test a robust Linear model using Log-transformed target and Target Encoding. This prevents the model from failing on raw Store IDs.

In [ ]:
ridge_scores = []
features = ["DayOfWeek", "Promo", "Year", "Month", "Store_TargetMean", "LogCompDist"]

for train_idx, test_idx in tscv.split(df):
    X_tr, X_te = build_features_v4(df.iloc[train_idx], df.iloc[test_idx])

    y_tr_log = np.log1p(X_tr["Sales"])
    y_te = X_te["Sales"]

    # Linear model needs specific features (numeric + ohe)
    cols = features + [
        c for c in X_tr.columns if "StoreType_" in c or "Assortment_" in c
    ]

    model = Pipeline([("scaler", StandardScaler()), ("reg", Ridge())])
    model.fit(X_tr[cols], y_tr_log)

    preds = np.expm1(model.predict(X_te[cols]))
    ridge_scores.append(rmspe(y_te, preds))

print(f"Ridge (Log + Target Enc) Avg RMSPE: {np.mean(ridge_scores) * 100:.2f}%")


## **5. Models Comparison**
Compare decision trees, ensembles, and gradients. All models use identical V4 (Target Encoded) features and Log-space training where applicable.

In [ ]:
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = config["train"]["random_state"]

models = {
    "Ridge (Linear)": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("reg", Ridge()),
        ]
    ),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=10,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=150,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        verbosity=-1,
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=150,
        max_depth=8,
        learning_rate=0.1,
        tree_method="hist",
        random_state=RANDOM_STATE,
    ),
}

# Determine base features
cols = [
    "DayOfWeek",
    "Promo",
    "Year",
    "Month",
    "WeekOfYear",
    "LogCompDist",
    "Store_TargetMean",
] + [
    c
    for c in df_train.columns
    if any(x in c for x in ["StoreType_", "Assortment_", "StateHoliday_"])
]

arena_results = []
arena_results.append({"Model": "Naive (Store Median)", **naive_avg})

for name, model in models.items():
    print(f"Testing {name}...")
    fold_metrics = []

    # Strictly split ONLY on df_cv
    for train_idx, test_idx in tscv.split(df_cv):
        X_tr, X_te = build_features_v4(df_cv.iloc[train_idx], df_cv.iloc[test_idx])
        y_tr_log = np.log1p(X_tr["Sales"])
        y_te = X_te["Sales"]

        # Train Log, predict Real
        model.fit(X_tr[cols], y_tr_log)
        preds = np.expm1(model.predict(X_te[cols]))

        fold_metrics.append(
            {
                "RMSPE": rmspe(y_te, preds),
                "RMSE": np.sqrt(mean_squared_error(y_te, preds)),
                "MAE": mean_absolute_error(y_te, preds),
                "R2": r2_score(y_te, preds),
            }
        )

    # Average metrics
    avg_metrics = pd.DataFrame(fold_metrics).mean().to_dict()
    arena_results.append({"Model": name, **avg_metrics})


In [ ]:
summary_df = pd.DataFrame(arena_results).sort_values("RMSPE")
summary_display = summary_df.copy()
summary_display["RMSPE (%)"] = summary_display["RMSPE"] * 100
summary_display = summary_display[["Model", "RMSPE (%)", "RMSE", "MAE", "R2"]]

# Visualization (RMSPE remains main sorting metric)
plt.figure(figsize=(14, 8))
ax = sns.barplot(data=summary_display, x="RMSPE (%)", y="Model", palette="magma")
plt.title("Model Comparison Leaderboard (Avg across Folds)")

for p in ax.patches:
    ax.annotate(
        f"{p.get_width():.2f}%",
        (p.get_width(), p.get_y() + p.get_height() / 2.0),
        ha="left",
        va="center",
        xytext=(5, 0),
        textcoords="offset points",
    )

plt.savefig(FIG_DIR / "06_full_model_leaderboard.png", dpi=300, bbox_inches="tight")
plt.show()

# Precise Table
display(
    summary_display.style.format(
        {"RMSPE (%)": "{:.2f}%", "RMSE": "{:.1f}", "MAE": "{:.1f}", "R2": "{:.3f}"}
    )
    .highlight_min(subset=["RMSPE (%)", "RMSE", "MAE"], color="lightgreen")
    .highlight_max(subset=["R2"], color="lightgreen")
)


<style type="text/css">
#T_b053d_row0_col1, #T_b053d_row0_col3, #T_b053d_row2_col4, #T_b053d_row3_col2 {
  background-color: lightgreen;
}
</style>
<table id="T_b053d">
  <thead>
    <tr>
      <th class="blank level0" >&nbsp;</th>
      <th id="T_b053d_level0_col0" class="col_heading level0 col0" >Model</th>
      <th id="T_b053d_level0_col1" class="col_heading level0 col1" >RMSPE (%)</th>
      <th id="T_b053d_level0_col2" class="col_heading level0 col2" >RMSE</th>
      <th id="T_b053d_level0_col3" class="col_heading level0 col3" >MAE</th>
      <th id="T_b053d_level0_col4" class="col_heading level0 col4" >R2</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th id="T_b053d_level0_row0" class="row_heading level0 row0" >3</th>
      <td id="T_b053d_row0_col0" class="data row0 col0" >Random Forest</td>
      <td id="T_b053d_row0_col1" class="data row0 col1" >22.58%</td>
      <td id="T_b053d_row0_col2" class="data row0 col2" >1431.6</td>
      <td id="T_b053d_row0_col3" class="data row0 col3" >957.1</td>
      <td id="T_b053d_row0_col4" class="data row0 col4" >0.790</td>
    </tr>
    <tr>
      <th id="T_b053d_level0_row1" class="row_heading level0 row1" >2</th>
      <td id="T_b053d_row1_col0" class="data row1 col0" >Decision Tree</td>
      <td id="T_b053d_row1_col1" class="data row1 col1" >23.04%</td>
      <td id="T_b053d_row1_col2" class="data row1 col2" >1475.4</td>
      <td id="T_b053d_row1_col3" class="data row1 col3" >985.9</td>
      <td id="T_b053d_row1_col4" class="data row1 col4" >0.777</td>
    </tr>
    <tr>
      <th id="T_b053d_level0_row2" class="row_heading level0 row2" >4</th>
      <td id="T_b053d_row2_col0" class="data row2 col0" >LightGBM</td>
      <td id="T_b053d_row2_col1" class="data row2 col1" >23.24%</td>
      <td id="T_b053d_row2_col2" class="data row2 col2" >1419.5</td>
      <td id="T_b053d_row2_col3" class="data row2 col3" >975.0</td>
      <td id="T_b053d_row2_col4" class="data row2 col4" >0.793</td>
    </tr>
    <tr>
      <th id="T_b053d_level0_row3" class="row_heading level0 row3" >5</th>
      <td id="T_b053d_row3_col0" class="data row3 col0" >XGBoost</td>
      <td id="T_b053d_row3_col1" class="data row3 col1" >23.57%</td>
      <td id="T_b053d_row3_col2" class="data row3 col2" >1418.8</td>
      <td id="T_b053d_row3_col3" class="data row3 col3" >963.9</td>
      <td id="T_b053d_row3_col4" class="data row3 col4" >0.792</td>
    </tr>
    <tr>
      <th id="T_b053d_level0_row4" class="row_heading level0 row4" >1</th>
      <td id="T_b053d_row4_col0" class="data row4 col0" >Ridge (Linear)</td>
      <td id="T_b053d_row4_col1" class="data row4 col1" >30.36%</td>
      <td id="T_b053d_row4_col2" class="data row4 col2" >2263.1</td>
      <td id="T_b053d_row4_col3" class="data row4 col3" >1263.1</td>
      <td id="T_b053d_row4_col4" class="data row4 col4" >0.476</td>
    </tr>
    <tr>
      <th id="T_b053d_level0_row5" class="row_heading level0 row5" >0</th>
      <td id="T_b053d_row5_col0" class="data row5 col0" >Naive (Store Median)</td>
      <td id="T_b053d_row5_col1" class="data row5 col1" >36.23%</td>
      <td id="T_b053d_row5_col2" class="data row5 col2" >2095.0</td>
      <td id="T_b053d_row5_col3" class="data row5 col3" >1505.7</td>
      <td id="T_b053d_row5_col4" class="data row5 col4" >0.552</td>
    </tr>
  </tbody>
</table>

## **Summary & Design Justfication**
We also conducted many other tests, but not shown or not fully shown here to keep the notebook leaner. Those tests have confirmed several decisions of ours:

#### **1. Why Log-Target?**
Predicting `Sales` (Linear) yields high residuals due to the range and skew. `log1p(Sales)` (Test C) stabilizes these residuals. Log transformation naturally optimizes for **RMSPE**, which treats percentage errors equally regardless of sales volume.

#### **2. Why Target Encoding?**
With 1,115 stores, Tree models and Linear models struggle to "index" store-specific baseline performance. Replacing `Store` ID with **Out-Of-Fold (OOF) Target Encoding** provides a high-signal sales baseline. This step alone dropped RMSPE by **~15 percentage points** in early tests. Without target encoding, models would treat store ID as a low-signal continuous variable, without much prediction power.

#### **3. Feature Selection**
We analyzed `CompetitionDistance` (Log), `CompAgeMonths`, and `Promo2` intervals. We found that OHE categorical signals for `StoreType` and `Assortment` are the strongest predictors of baseline volume, while `Promo` and `DayOfWeek` drive the daily variance.

#### **4. Architecture Observation & Choice**
In this "Arena" (using default/lite parameters), **Random Forest** and **Decision Trees** proved surprisingly robust. **XGBoost** and **LightGBM** trailed slightly, likely due to under-tuning of regularization (regularization defaults in GBMs are more aggressive).

**Outcome**: While Random Forest is a strong ensemble baseline, we will try tuning these models to see if the gains are significant and if the ordering changes.
